# 🏥 Medical Insurance Cost Prediction — Project Walkthrough

**A complete end-to-end guide** — from raw CSV to live predictions.

This notebook walks through every layer of the project:
- What each folder and file does
- The exact functions that transform data step by step
- How modules interact and pass data between them
- Key code snippets with explanations
- How to run and test everything

---

## 📁 Project Structure at a Glance

```
medical-insurance-cost-prediction/
│
├── src/                    ← Python ML library (the brain)
│   ├── utils.py            ← Foundation: paths, logger, data loading
│   ├── preprocessing.py    ← Clean and split raw data
│   ├── feature_engineering.py  ← Add 8 domain-driven features
│   ├── train.py            ← Train 10 models end-to-end
│   ├── evaluate.py         ← Measure and rank models
│   ├── predict.py          ← Load model and make predictions
│   └── visualization.py    ← Generate all plots
│
├── frontend/               ← Next.js 16 dashboard (UI)
├── main.py                 ← FastAPI backend (JSON API)
├── data/insurance.csv      ← Raw dataset (1,338 rows)
├── models/best_model.pkl   ← Saved winning pipeline
└── outputs/                ← Generated plots, metrics, reports
```

---
## 🔄 End-to-End Data Flow

```
insurance.csv
    ↓  utils.load_raw_data()
    ↓  preprocessing.clean_dataframe()      remove dup, validate, standardise
    ↓  feature_engineering.engineer_features()  add 8 new columns
    ↓  preprocessing.split_data()           80/20 stratified split
    ↓  preprocessing.build_preprocessor()   StandardScaler + OneHotEncoder
    ↓  train.train_all_models()             10 × Pipeline.fit()
    ↓  evaluate.select_best_model()         composite rank + CV tie-break
    ↓  joblib.dump(best_model.pkl)          save winning pipeline
    ↓  predict.predict_charges({...})       live inference
```

---
## 📦 Step 0 — Setup
Run from the project root. All imports assume `PYTHONPATH=.`

In [ ]:
import sys
sys.path.insert(0, '..')   # so 'src' is importable from notebooks/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('Setup complete ✓')

---
## 🗂️ Module 1 — `src/utils.py`
**Role:** Foundation layer. Every other module imports from here.

**What it provides:**
- All project-wide file paths (resolved automatically from file location)
- Shared constants (`TARGET_COLUMN`, `RANDOM_STATE`, `BMI_OBESE_THRESHOLD`)
- `load_raw_data()` — loads insurance.csv with helpful error if missing
- `get_logger()` — consistent timestamped logging across all modules
- `save_json()` / `save_markdown()` — I/O helpers
- `ensure_output_dirs()` — creates outputs/ subfolders before writing

**Key design:** Nothing in `utils.py` imports from other `src/` files → zero circular dependencies.

In [ ]:
from src.utils import (
    ROOT_DIR, DATA_DIR, MODELS_DIR, OUTPUTS_DIR,
    TARGET_COLUMN, RANDOM_STATE, BMI_OBESE_THRESHOLD,
    load_raw_data, get_logger
)

# All paths are resolved relative to the project root — works on any machine
print(f'Project root : {ROOT_DIR}')
print(f'Data dir     : {DATA_DIR}')
print(f'Target column: {TARGET_COLUMN}')
print(f'Random state : {RANDOM_STATE}')
print(f'Obese BMI    : {BMI_OBESE_THRESHOLD}')

# Load raw data
raw = load_raw_data()
print(f'\nRaw data shape: {raw.shape}')
raw.head(3)

---
## 🧹 Module 2 — `src/preprocessing.py`
**Role:** Clean raw data and build the sklearn transformation pipeline.

### Functions & what they do:

| Function | Input | Output | Key action |
|---|---|---|---|
| `remove_duplicates(df)` | raw df | df | Drops 1 exact duplicate row |
| `standardise_categoricals(df)` | df | df | Lowercase + strip all strings |
| `validate_ranges(df)` | df | df | Nullifies impossible values (age>120 etc) |
| `handle_missing(df)` | df | df | Median fill numeric, mode fill categorical |
| `clean_dataframe(df)` | raw df | clean df | Calls all 4 above in sequence |
| `build_preprocessor(...)` | config | ColumnTransformer | Builds sklearn transform pipeline |
| `split_data(df)` | clean df | X_train,X_test,y_train,y_test | Stratified 80/20 split |

In [ ]:
from src.preprocessing import (
    remove_duplicates, standardise_categoricals,
    validate_ranges, clean_dataframe, split_data
)

# Step 1: Remove the 1 known duplicate row
# Before: 1338 rows   After: 1337 rows
deduped = remove_duplicates(raw)
print(f'After dedup: {deduped.shape[0]} rows (removed {raw.shape[0] - deduped.shape[0]})')

# Step 2: Lowercase all categoricals so 'Male' == 'male' == 'MALE'
normalised = standardise_categoricals(deduped)
print(f'Unique sex values: {normalised["sex"].unique()}')
print(f'Unique smoker values: {normalised["smoker"].unique()}')

# Step 3: Run full cleaning pipeline
cleaned = clean_dataframe(raw)
print(f'\nCleaned shape: {cleaned.shape}')
print(f'Missing values: {cleaned.isna().sum().sum()}')

In [ ]:
# The ColumnTransformer: different treatment for numeric vs categorical
#
# Numeric  → SimpleImputer(median) + StandardScaler  (for linear/SVR models)
# Categorical → SimpleImputer(mode) + OneHotEncoder(drop='first')
#
# Key: build_preprocessor(scale_numeric=True)  → linear models
#      build_preprocessor(scale_numeric=False) → tree models

from src.preprocessing import build_preprocessor

# Example: preprocessor for a linear model (with scaling)
preprocessor_scaled = build_preprocessor(
    scale_numeric=True,
    numeric_features=['age', 'bmi', 'children'],
    categorical_features=['sex', 'smoker', 'region']
)
print('Preprocessor steps:')
for name, transformer, cols in preprocessor_scaled.transformers:
    print(f'  {name}: {[s[0] for s in transformer.steps]} on {cols}')

In [ ]:
# Stratified split — preserves the 20% smoker ratio in both train and test
# Stratification key = smoker_status + charge_quartile (not just smoker alone)
# This prevents train/test imbalance on the most important predictor

from src.feature_engineering import engineer_features

enriched = engineer_features(cleaned)
X_train, X_test, y_train, y_test = split_data(enriched, test_size=0.20)

print(f'Train: {len(X_train)} rows   Test: {len(X_test)} rows')
print(f'Train smoker %: {(X_train["smoker"]=="yes").mean()*100:.1f}%')
print(f'Test  smoker %: {(X_test["smoker"]=="yes").mean()*100:.1f}%')
print(f'\ny_train range: ${y_train.min():,.0f} – ${y_train.max():,.0f}')
print(f'y_test  range: ${y_test.min():,.0f} – ${y_test.max():,.0f}')

---
## ⚙️ Module 3 — `src/feature_engineering.py`
**Role:** Create 8 new columns from the 6 raw inputs. No target leakage.

**Why this matters:** The raw dataset has nearly linear relationships once
the interaction terms are encoded. Without them, tree models would beat
linear ones. With them, Lasso wins.

### Feature map:

| New Feature | Built From | Type | Motivation |
|---|---|---|---|
| `age_group` | `age` | categorical | young/middle_age/senior — nonlinear age effect |
| `bmi_category` | `bmi` | categorical | WHO buckets — clinical risk |
| `is_obese` | `bmi` | binary | 1 if BMI ≥ 30 |
| `smoker_obese` | `smoker` × `is_obese` | binary | **KEY** — highest charge group |
| `age_bmi` | `age` × `bmi` / 1000 | numeric | Compound metabolic risk |
| `has_children` | `children > 0` | binary | Cleaner than raw count |
| `family_size` | `children` bucketed | categorical | individual/small/large |
| `age_smoker` | `age` × smoker_flag | numeric | Older smokers pay far more |

In [ ]:
from src.feature_engineering import (
    engineer_features,
    add_smoker_obese_interaction,
    add_age_smoker_interaction,
    get_all_feature_groups
)

# Before engineering: 14 cols (after cleaning target column is still there)
print(f'Before: {cleaned.shape[1]} columns → {list(cleaned.columns)}')

# After engineering: +8 new columns
enriched = engineer_features(cleaned)
new_cols = [c for c in enriched.columns if c not in cleaned.columns]
print(f'\nAfter: {enriched.shape[1]} columns')
print(f'New columns added: {new_cols}')

In [ ]:
# The most important feature: smoker_obese interaction
# smokers with BMI >= 30 have dramatically higher charges

print('Average charges by smoker_obese group:')
print(enriched.groupby('smoker_obese')['charges'].agg(['mean','count','median']))
print()

# age_smoker: older smokers face compounding costs
print('Sample age_smoker values (age x smoker_flag):')
print(enriched[['age','smoker','age_smoker']].head(8).to_string(index=False))

In [ ]:
# After engineering, feature groups are derived automatically
# This feeds directly into build_preprocessor() in training

X_sample = enriched.drop(columns=['charges'])
num_feats, cat_feats = get_all_feature_groups(X_sample)

print(f'Numeric features  ({len(num_feats)}): {num_feats}')
print(f'Categorical features ({len(cat_feats)}): {cat_feats}')

---
## 🏋️ Module 4 — `src/train.py`
**Role:** Orchestrates the entire training run. Calls all other modules in sequence.

### Key functions:

| Function | What it does |
|---|---|
| `prepare_data()` | load → clean → engineer → split — returns X_train,X_test,y_train,y_test |
| `_get_model_registry()` | Returns dict of {name: estimator} for all 10 models |
| `_build_pipeline(estimator, ...)` | Wraps estimator in sklearn Pipeline with preprocessor |
| `train_all_models(...)` | Loops registry → fits → evaluates → cross-validates each |
| `save_best_model(...)` | Serialises winning pipeline to models/best_model.pkl |
| `run_training_pipeline()` | Master orchestrator — calls everything above |

### Why a Pipeline matters:
```python
Pipeline(steps=[
    ('preprocessor', ColumnTransformer(...)),  # scaling + OHE
    ('regressor',    LassoCV(...))             # model
])
```
The preprocessor is **baked into** the pipeline. When you save `best_model.pkl`,
you save both the scaler AND the model together — no drift, no missing files.

In [ ]:
# Model registry — 10 estimators with tuned hyperparameters
# Note: RidgeCV and LassoCV automatically tune their alpha via cross-validation

from sklearn.linear_model import LassoCV, LinearRegression, RidgeCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

RANDOM_STATE = 42

# Linear models — need StandardScaler (scale_numeric=True)
ridge = RidgeCV(
    alphas=[0.01, 0.1, 1.0, 10.0, 100.0, 500.0, 1000.0],
    scoring='neg_root_mean_squared_error'
)
# RidgeCV.alpha_ is set AFTER fitting — it holds the CV-chosen value

lasso = LassoCV(
    alphas=100,          # 100 candidates on the regularisation path
    cv=5,
    max_iter=50_000,
    random_state=RANDOM_STATE
)
# LassoCV.alpha_ similarly set after fitting

print('RidgeCV: alpha selected by leave-one-out CV')
print('LassoCV: alpha selected by 5-fold CV across 100 candidates')
print('Both guarantee data-driven regularisation strength, not guesswork')

In [ ]:
# The Pipeline wraps preprocessor + estimator
# SCALED_MODELS get StandardScaler; tree models skip it

from sklearn.pipeline import Pipeline
from src.preprocessing import build_preprocessor, split_data
from src.feature_engineering import engineer_features, get_all_feature_groups
from src.preprocessing import clean_dataframe
from src.utils import load_raw_data

# Reproduce the exact pipeline used in training
raw = load_raw_data()
cleaned = clean_dataframe(raw)
enriched = engineer_features(cleaned)
X_train, X_test, y_train, y_test = split_data(enriched)
num_feats, cat_feats = get_all_feature_groups(X_train)

preprocessor = build_preprocessor(
    scale_numeric=True,    # True for linear models
    numeric_features=num_feats,
    categorical_features=cat_feats
)

lasso_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LassoCV(alphas=100, cv=5, max_iter=50_000, random_state=42))
])

lasso_pipeline.fit(X_train, y_train)
chosen_alpha = lasso_pipeline.named_steps['regressor'].alpha_
print(f'LassoCV selected alpha = {chosen_alpha:.4f}')
print(f'Input shape → {X_train.shape}  |  After OHE → {lasso_pipeline["preprocessor"].transform(X_train).shape}')

---
## 📊 Module 5 — `src/evaluate.py`
**Role:** Measure every model and select the winner.

### Functions:

| Function | What it does |
|---|---|
| `compute_metrics(y_true, y_pred)` | Returns MAE, RMSE, R² on dollar scale |
| `evaluate_model(pipeline, ...)` | Fits + predicts + computes metrics in one call |
| `run_cross_validation(pipeline, X, y)` | 5-fold CV → R² mean/std per model |
| `compute_learning_curve(pipeline, X, y)` | Train vs validation R² at increasing data sizes |
| `select_best_model(metrics_df, cv_data)` | Composite rank + CV tie-break → winner name |

### Selection formula:
```
composite = 0.45 × rmse_rank + 0.35 × mae_rank + 0.20 × r2_rank
```
Models within 0.5 composite points are tied → broken by CV R².

In [ ]:
from src.evaluate import compute_metrics, evaluate_model, run_cross_validation, select_best_model

# Evaluate the Lasso pipeline we built above
y_pred = lasso_pipeline.predict(X_test)
metrics = compute_metrics(y_test.values, y_pred)

print('Lasso Regression — Test Set Metrics:')
print(f'  RMSE : ${metrics["rmse"]:>10,.2f}   (average prediction error)')
print(f'  MAE  : ${metrics["mae"]:>10,.2f}   (median-style error)')
print(f'  R²   : {metrics["r2"]:>10.4f}   (variance explained)')

In [ ]:
# Cross-validation gives a more reliable estimate than a single test split
# It trains/tests on 5 different folds and averages the scores

from sklearn.base import clone
from src.preprocessing import build_preprocessor

cv_pipeline = Pipeline(steps=[
    ('preprocessor', build_preprocessor(True, num_feats, cat_feats)),
    ('regressor', LassoCV(alphas=100, cv=5, max_iter=50_000, random_state=42))
])

cv_result = run_cross_validation(cv_pipeline, X_train, y_train)
print(f'5-fold CV R²: {cv_result["r2_mean"]:.4f} ± {cv_result["r2_std"]:.4f}')
print(f'Per-fold R²:  {[round(v,4) for v in cv_result["r2_folds"]]}')
print(f'CV RMSE:      ${cv_result["rmse_mean"]:,.0f} ± ${cv_result["rmse_std"]:,.0f}')

In [ ]:
# Load the actual comparison results from the last training run
import pandas as pd
from src.utils import METRICS_DIR

comparison = pd.read_csv(METRICS_DIR / 'comparison.csv')
display_cols = ['model', 'rmse', 'mae', 'r2', 'train_time_sec']
df_show = comparison[display_cols].sort_values('rmse').reset_index(drop=True)
df_show.index += 1
df_show['rmse'] = df_show['rmse'].map('${:,.0f}'.format)
df_show['mae']  = df_show['mae'].map('${:,.0f}'.format)
df_show['r2']   = df_show['r2'].map('{:.4f}'.format)
print(df_show.to_string())

---
## 🔮 Module 6 — `src/predict.py`
**Role:** Load the saved model and make predictions in production.

**Key design:** Applies the EXACT same cleaning + feature engineering as training.
If you skip `engineer_features()` at inference time, the model gets wrong features.

### How it works:
```
raw dict input
    ↓  standardise strings (lowercase, strip)
    ↓  cast numeric types
    ↓  engineer_features()  ← same 8 features as training
    ↓  align columns to training feature list
    ↓  pipeline.predict()   ← preprocessor + LassoCV
    ↓  clip to >= 0
    → predicted USD charge
```

### Caching:
The model bundle is cached with `@lru_cache` — the pkl file is only
loaded from disk once per process, keeping the FastAPI server fast.

In [ ]:
from src.predict import predict_charges, predict_batch, get_model_info

# Single prediction
profiles = [
    {'age':22, 'sex':'male',   'bmi':22.0, 'children':0, 'smoker':'no',  'region':'northeast'},
    {'age':35, 'sex':'female', 'bmi':28.5, 'children':2, 'smoker':'no',  'region':'northwest'},
    {'age':45, 'sex':'male',   'bmi':33.0, 'children':1, 'smoker':'yes', 'region':'southeast'},
    {'age':60, 'sex':'female', 'bmi':35.0, 'children':0, 'smoker':'yes', 'region':'southwest'},
]
labels = [
    'Young healthy male',
    'Middle-age female, 2 kids',
    'Obese smoker male',
    'Senior obese smoker female'
]

print('Example predictions:')
print('-' * 45)
for label, profile in zip(labels, profiles):
    charge = predict_charges(profile)
    print(f'  {label:<30}: ${charge:>10,.2f}')

In [ ]:
# Batch prediction — pass a DataFrame
import pandas as pd

batch_df = pd.DataFrame(profiles)
results = predict_batch(batch_df)
batch_df['predicted_charges'] = results.map('${:,.2f}'.format)
print(batch_df[['age','sex','bmi','smoker','predicted_charges']].to_string(index=False))

In [ ]:
# Inspect what's inside the saved model bundle
info = get_model_info()
print(f'Model name       : {info["model_name"]}')
print(f'Target column    : {info["target_column"]}')
print(f'Numeric features : {info["numeric_features"]}')
print(f'Categorical feat : {info["categorical_features"]}')
print(f'Total features   : {info["n_features_total"]}')

---
## 🌐 Module 7 — `main.py` (FastAPI Backend)
**Role:** Exposes the ML pipeline as a JSON REST API.

**Runs on:** `http://localhost:8000`  
**Interactive docs:** `http://localhost:8000/docs`

### How the API processes a prediction request:
```
POST /api/predict  →  PredictRequest (Pydantic validates types + ranges)
    ↓  src.predict.predict_charges(inp)      # calls the ML pipeline
    ↓  _risk_flags(inp)                      # adds context warnings
    ↓  dataset stats for benchmarking
    → JSON: { predicted_charges, formatted, model_used, risk_flags, context }
```

### CORS configuration:
```python
allow_origins=['http://localhost:3000']   # Next.js dev server
```
This is what allows the browser (Next.js at port 3000) to call the API
(FastAPI at port 8000) without being blocked by browser security.

### In-memory caching:
```python
_cache: dict = {}   # dataset, metrics, CV data cached on first request
```
The CSV and JSON files are read once and stored in memory — subsequent
requests return instantly without hitting the filesystem.

In [ ]:
# You can test the API directly with Python's requests library
# Make sure uvicorn is running first: uvicorn main:app --reload --port 8000

try:
    import requests
    
    resp = requests.post(
        'http://localhost:8000/api/predict',
        json={'age':45,'sex':'male','bmi':33.0,'children':1,'smoker':'yes','region':'southeast'},
        timeout=3
    )
    result = resp.json()
    print(f'Prediction : {result["formatted"]}')
    print(f'Model used : {result["model_used"]}')
    print(f'Risk flags : {result["risk_flags"]}')
except Exception as e:
    print(f'API not running (start with: uvicorn main:app --reload --port 8000)')
    print(f'Error: {e}')

---
## 🖥️ Module 8 — `frontend/` (Next.js Dashboard)
**Role:** Visual interface for everything the API exposes.

**Runs on:** `http://localhost:3000`

### How each page gets its data:

| Page file | Fetches from |
|---|---|
| `app/page.tsx` | `GET /api/dataset/summary` + `GET /api/metrics` |
| `app/dataset/page.tsx` | `GET /api/dataset/summary` + `GET /api/dataset/sample` |
| `app/eda/page.tsx` | `GET /api/eda-figures` → renders PNG URLs |
| `app/metrics/page.tsx` | `GET /api/metrics` + `GET /api/eval-figures/{stem}` |
| `app/predict/page.tsx` | `POST /api/predict` on form submit |
| `app/about/page.tsx` | Static (no API call) |

### Key components:

**`ChartCard.tsx`** — every plot is wrapped in this:
```tsx
// Provides minimise / maximise toggle on each chart
<ChartCard title='Charges Distribution' defaultMinimized={false}>
    <img src='http://localhost:8000/outputs/figures/eda/charges_distribution.png' />
</ChartCard>
```

**`ImageModal.tsx`** — click any image to open fullscreen:
```tsx
// Opens a fullscreen overlay with keyboard navigation
<ImageModal images={edaImages} initialIndex={0} onClose={() => setModal(false)} />
```

**Why `<img>` not `<Image>` from Next.js:**  
FastAPI serves PNGs with dynamic paths at runtime. Next.js `<Image>` needs
static/known paths at build time. Raw `<img>` works here; we configure
`next.config.ts` to document the intent.

---
## 🔗 Module Interaction Map

How modules call each other:

```
src/utils.py          ← imported by EVERYTHING (paths, logger, constants)
    ↑
src/preprocessing.py  ← imported by train.py, predict.py
    ↑
src/feature_engineering.py  ← imported by train.py, predict.py
    ↑
src/evaluate.py       ← imported by train.py
    ↑
src/visualization.py  ← imported by train.py
    ↑
src/train.py          ← calls all src/ modules, saves best_model.pkl
    ↑
src/predict.py        ← loads best_model.pkl, used by main.py
    ↑
main.py               ← FastAPI, imports predict.py + utils.py directly
    ↑
frontend/             ← Next.js, communicates with main.py via HTTP only
```

**Dependency rule:** Each layer only imports from the layer below it.
No circular dependencies. `utils.py` imports nothing from `src/`.

---
## 📈 Key Data Insight — Why Lasso Wins
The dataset has one dominant predictor (smoker status). Let's verify this.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Why Feature Engineering Makes Linear Models Win', fontsize=13, fontweight='bold')

# 1. Charges by smoker status
ax = axes[0]
for s, color, label in [('yes','#ef4444','Smoker'), ('no','#22c55e','Non-smoker')]:
    subset = enriched[enriched['smoker']==s]['charges']
    ax.hist(subset, bins=30, alpha=0.6, color=color, label=label)
ax.set_title('Charges by Smoker Status')
ax.set_xlabel('Annual Charge ($)')
ax.legend()
ax.yaxis.set_visible(False)

# 2. smoker_obese interaction
ax = axes[1]
groups = enriched.groupby('smoker_obese')['charges'].mean()
bars = ax.bar(['Non-obese/Non-smoker\n(smoker_obese=0)', 'Obese Smoker\n(smoker_obese=1)'],
              groups.values, color=['#6366f1','#ef4444'], alpha=0.8)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
            f'${bar.get_height():,.0f}', ha='center', fontsize=9)
ax.set_title('Key Interaction: smoker_obese')
ax.set_ylabel('Mean Charges ($)')
ax.yaxis.set_visible(False)

# 3. age_smoker vs charges — shows linear relationship
ax = axes[2]
ax.scatter(enriched['age_smoker'], enriched['charges'],
           alpha=0.3, s=15, c='#6366f1')
ax.set_title('age_smoker vs Charges\n(linear relationship → Lasso wins)')
ax.set_xlabel('age_smoker (age × smoker_flag)')
ax.set_ylabel('Charges ($)')

plt.tight_layout()
plt.show()

In [ ]:
# Lasso coefficient importance — which features it selected
import joblib
from src.utils import MODELS_DIR
from src.preprocessing import get_feature_names_out

bundle = joblib.load(MODELS_DIR / 'best_model.pkl')
pipeline = bundle['model']
regressor = pipeline.named_steps['regressor']
preprocessor = pipeline.named_steps['preprocessor']

feature_names = get_feature_names_out(preprocessor)
coefs = regressor.coef_

# Top 10 most important features by absolute coefficient
coef_df = pd.DataFrame({'feature': feature_names, 'coefficient': coefs})
coef_df = coef_df.reindex(coef_df['coefficient'].abs().sort_values(ascending=False).index)
top10 = coef_df.head(10)

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#ef4444' if v > 0 else '#6366f1' for v in top10['coefficient']]
ax.barh(top10['feature'], top10['coefficient'], color=colors, alpha=0.8)
ax.set_title('Lasso Regression — Top 10 Feature Coefficients')
ax.set_xlabel('Coefficient value')
ax.axvline(0, color='white', linewidth=0.8)
plt.tight_layout()
plt.show()
print('\nZero coefficients (features Lasso eliminated):',
      (coefs == 0).sum(), 'out of', len(coefs))

---
## ✅ How to Run Everything

Open **3 separate terminals** from the project root:

### Terminal 1 — Train (one time)
```powershell
# Windows
$env:PYTHONPATH = '.'
python -m src.train

# macOS / Linux
PYTHONPATH=. python -m src.train
```

### Terminal 2 — FastAPI backend
```bash
uvicorn main:app --reload --port 8000
# → http://localhost:8000/docs
```

### Terminal 3 — Next.js dashboard
```bash
cd frontend
npm run dev
# → http://localhost:3000
```

### Run tests
```powershell
$env:PYTHONPATH = '.'
pytest tests/ -v
# 52 tests — all passing
```

---
## 🧪 Testing Structure

| File | Tests | What's verified |
|---|---|---|
| `tests/test_preprocessing.py` | 20 | `clean_dataframe`, `split_data`, `build_preprocessor`, range validation, stratification |
| `tests/test_predict.py` | 20 | `predict_charges`, `predict_batch`, `get_model_info`, edge cases (smoker obese, senior) |
| `tests/test_api.py` | 12 | All FastAPI endpoints — status codes, field validation, response shapes |

---
## 📝 Summary

| Layer | File(s) | Language | Role |
|---|---|---|---|
| Data foundation | `src/utils.py` | Python | Paths, logger, constants, I/O |
| Data cleaning | `src/preprocessing.py` | Python | Clean, validate, split, OHE pipeline |
| Feature creation | `src/feature_engineering.py` | Python | 8 interaction + domain features |
| Model training | `src/train.py` | Python | Train 10 models, save best |
| Model evaluation | `src/evaluate.py` | Python | Metrics, CV, composite ranking |
| Inference | `src/predict.py` | Python | Load pkl, single + batch predict |
| Visualisation | `src/visualization.py` | Python | 48 matplotlib/seaborn plots |
| REST API | `main.py` | Python | FastAPI — 9 JSON endpoints |
| UI | `frontend/` | TypeScript | Next.js 16 — 6 pages, shadcn/ui |

**Best model:** Lasso Regression — RMSE **$4,493** · R² **0.8857** · CV-tuned alpha